# Customer Support Email Agent with GPT-4o and Commune

This notebook walks through building a **production-ready customer support agent** that:

- Reads incoming support emails from a [Commune](https://commune.email) inbox
- Classifies each ticket by type and priority using GPT-4o
- Drafts contextually appropriate replies
- Sends those replies automatically via the Commune API

**What you'll learn:**
- How to define tools in OpenAI's function-calling JSON schema
- How to implement the full `tool_calls` → function execution → `tool` message loop
- How to build a robust multi-turn agent with error handling
- Production patterns: polling loops, rate limiting, graceful shutdown

**Prerequisites:**
Set the environment variables `OPENAI_API_KEY` and `COMMUNE_API_KEY`.


## 1. Setup


In [ ]:
%pip install -q openai commune-mail


In [ ]:
import os
import json
import time
import threading
from openai import OpenAI
from commune import Commune

openai_client  = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
commune_client = Commune(api_key=os.environ["COMMUNE_API_KEY"])

MODEL = "gpt-4o"
print("Clients initialised.")


## 2. Email Tool Functions

These Python functions wrap the Commune SDK and serve as the "hands"
the agent uses to interact with the real inbox.


In [ ]:
def read_inbox(limit: int = 10, unread_only: bool = False) -> dict:
    """Fetch recent emails from the Commune support inbox."""
    emails = commune_client.emails.list(limit=limit, unread_only=unread_only)
    return {
        "emails": [
            {
                "id":           e["id"],
                "from":         e["from_address"],
                "subject":      e["subject"],
                "preview":      e["body"][:400],
                "received_at":  e["received_at"],
                "read":         e["read"],
            }
            for e in emails
        ],
        "count": len(emails),
    }


def get_email(email_id: str) -> dict:
    """Get the complete body of a single email."""
    e = commune_client.emails.get(email_id)
    return {
        "id":          e["id"],
        "from":        e["from_address"],
        "subject":     e["subject"],
        "body":        e["body"],
        "received_at": e["received_at"],
    }


def search_emails(query: str, limit: int = 10) -> dict:
    """Full-text search across the inbox."""
    results = commune_client.emails.search(query=query, limit=limit)
    return {
        "results": [
            {
                "id":          e["id"],
                "from":        e["from_address"],
                "subject":     e["subject"],
                "preview":     e["body"][:400],
                "received_at": e["received_at"],
            }
            for e in results
        ],
        "count": len(results),
    }


def send_email(to: str, subject: str, body: str) -> dict:
    """Send an email reply via Commune."""
    result = commune_client.emails.send(to=to, subject=subject, body=body)
    return {"status": "sent", "message_id": result.get("id")}


def send_sms(to: str, body: str) -> dict:
    """Send an SMS alert for urgent tickets (Commune SMS)."""
    result = commune_client.sms.send(to=to, body=body)
    return {"status": "sent", "message_id": result.get("id")}


# Registry used by the agent executor
TOOL_REGISTRY = {
    "read_inbox":    read_inbox,
    "get_email":     get_email,
    "search_emails": search_emails,
    "send_email":    send_email,
    "send_sms":      send_sms,
}

print("Tool functions:", list(TOOL_REGISTRY))


## 3. Tool Definitions (OpenAI Function-Calling Schema)

OpenAI's `tools` list uses the `{"type": "function", "function": {...}}` wrapper.
The `parameters` field follows JSON Schema.


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "read_inbox",
            "description": (
                "Read recent emails from the Commune support inbox. "
                "Use this to check for new messages, triage tickets, or get an overview "
                "of pending support requests."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "limit": {
                        "type": "integer",
                        "description": "Number of emails to return (1–50). Default 10.",
                    },
                    "unread_only": {
                        "type": "boolean",
                        "description": "Return only unread emails.",
                    },
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_email",
            "description": (
                "Retrieve the full body of a specific email by its ID. "
                "Call this after read_inbox to read the complete content of a message."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "email_id": {
                        "type": "string",
                        "description": "The unique ID of the email.",
                    },
                },
                "required": ["email_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_emails",
            "description": (
                "Search the inbox by keyword, topic, or sender address. "
                "Useful for finding all tickets related to a specific issue or customer."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search terms or keywords.",
                    },
                    "limit": {
                        "type": "integer",
                        "description": "Max results to return (default 10).",
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": (
                "Send an email reply to a customer via Commune. "
                "Always base the reply on the actual email content — do not fabricate details."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {
                        "type": "string",
                        "description": "Recipient email address.",
                    },
                    "subject": {
                        "type": "string",
                        "description": "Email subject line.",
                    },
                    "body": {
                        "type": "string",
                        "description": "Plain-text email body.",
                    },
                },
                "required": ["to", "subject", "body"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "send_sms",
            "description": (
                "Send an SMS alert via Commune for urgent tickets that need immediate human attention. "
                "Use sparingly — only for P0/P1 severity issues."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {
                        "type": "string",
                        "description": "Recipient phone number in E.164 format, e.g. +15551234567.",
                    },
                    "body": {
                        "type": "string",
                        "description": "SMS message text (max 160 chars).",
                    },
                },
                "required": ["to", "body"],
            },
        },
    },
]

print(f"{len(tools)} tools registered.")


## 4. Agent Implementation

The agent loop follows the standard OpenAI function-calling pattern:

1. Call `chat.completions.create` with `tools` and `tool_choice="auto"`
2. If the model returns `finish_reason == "tool_calls"`, execute each call
3. Append the function outputs as `{"role": "tool", ...}` messages
4. Loop until `finish_reason == "stop"`


In [ ]:
SYSTEM_PROMPT = """\
You are an expert customer support agent with access to a Commune email inbox.
Your job is to:
  1. Triage incoming support emails by urgency and category.
  2. Draft professional, empathetic replies.
  3. Send those replies via the send_email tool.
  4. Escalate critical issues (outages, billing failures, data loss) via SMS.

Guidelines:
- Always read the full email body before drafting a reply.
- Address the customer by their first name when possible.
- Be concise — support replies should be under 200 words.
- If you cannot resolve an issue, acknowledge it and promise a follow-up within 24 hours.
- Never make up product details — if unsure, say so.
"""


def execute_tool_call(tool_call) -> str:
    """Execute a single OpenAI tool call and return the result as a JSON string."""
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    fn = TOOL_REGISTRY.get(name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool: {name}"})

    try:
        result = fn(**args)
        return json.dumps(result)
    except Exception as exc:
        return json.dumps({"error": str(exc)})


def run_support_agent(user_message: str, verbose: bool = True) -> str:
    """Run the GPT-4o support agent for a single user request."""
    messages = [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": user_message},
    ]

    iteration = 0
    max_iterations = 20   # safety guard against infinite loops

    while iteration < max_iterations:
        iteration += 1

        response = openai_client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        choice = response.choices[0]
        message = choice.message

        if verbose:
            print(f"[iteration {iteration}] finish_reason={choice.finish_reason}")

        # Always append the assistant's message to history
        messages.append(message)

        # ── Final answer ─────────────────────────────────────────────────
        if choice.finish_reason == "stop":
            return message.content or "(no content)"

        # ── Tool calls ───────────────────────────────────────────────────
        if choice.finish_reason == "tool_calls" and message.tool_calls:
            for tool_call in message.tool_calls:
                if verbose:
                    print(f"  → {tool_call.function.name}({tool_call.function.arguments[:120]})")

                result_str = execute_tool_call(tool_call)

                if verbose:
                    print(f"  ← {result_str[:200]}{'...' if len(result_str) > 200 else ''}")

                messages.append({
                    "role":         "tool",
                    "tool_call_id": tool_call.id,
                    "content":      result_str,
                })
            continue

        # Unexpected finish reason
        raise RuntimeError(f"Unexpected finish_reason: {choice.finish_reason}")

    raise RuntimeError("Agent exceeded max iterations — possible loop.")


print("Support agent ready.")


## 5. Demo — Process the Inbox

Run the agent against a real inbox. It will read, classify, and reply to
support emails autonomously.


In [ ]:
# Triage and classify unread tickets
result = run_support_agent(
    "Check the inbox for unread support emails. For each one:\n"
    "  1. Classify it as one of: billing, technical, feature-request, general.\n"
    "  2. Assign a priority: P0 (outage/data loss), P1 (broken feature), P2 (question).\n"
    "  3. Draft and send a reply that acknowledges the issue and sets expectations.\n"
    "After processing all tickets, give me a summary table."
)

print("\n" + "=" * 60)
print(result)


In [ ]:
# Follow up on a specific topic
result = run_support_agent(
    "Search for any emails mentioning 'refund' or 'charge'. "
    "Read each one in full, then send a reply that: "
    "(1) apologises for any billing confusion, "
    "(2) explains our 30-day no-questions-asked refund policy, "
    "(3) tells them to reply to this email to initiate the refund. "
    "Use the customer's name in the greeting."
)

print("\n" + "=" * 60)
print(result)


## 6. Production Patterns

### 6a. Polling Loop with Rate Limiting

Poll the inbox on a schedule. A simple exponential backoff prevents hammering
the API if something goes wrong.


In [ ]:
import threading
import time
import math

POLL_INTERVAL = 300      # seconds between polls (5 minutes)
MAX_BACKOFF   = 1800     # max wait on repeated errors (30 minutes)

_stop_flag = threading.Event()


def support_polling_loop(
    escalation_phone: str = "+15550001234",
    verbose: bool = True,
):
    """Continuously poll the support inbox and process new tickets."""
    consecutive_errors = 0

    while not _stop_flag.is_set():
        wait = min(POLL_INTERVAL * (2 ** consecutive_errors), MAX_BACKOFF)
        if consecutive_errors > 0:
            print(f"[poll] Backing off {wait}s after {consecutive_errors} error(s).")
            _stop_flag.wait(timeout=wait)
            if _stop_flag.is_set():
                break

        ts = time.strftime("%Y-%m-%d %H:%M:%S")
        print(f"[poll] {ts} — checking inbox ...")

        try:
            summary = run_support_agent(
                "Check for unread emails received in the last 10 minutes. "
                f"For critical P0 issues, send an SMS alert to {escalation_phone}. "
                "Reply to all tickets. If the inbox is empty, say 'No new tickets.'",
                verbose=verbose,
            )
            print(f"[poll] Result: {summary[:300]}")
            consecutive_errors = 0
        except Exception as exc:
            consecutive_errors += 1
            print(f"[poll] ERROR: {exc}")

        _stop_flag.wait(timeout=POLL_INTERVAL)

    print("[poll] Stopped.")


def start_support_agent(escalation_phone: str = "+15550001234"):
    _stop_flag.clear()
    t = threading.Thread(
        target=support_polling_loop,
        kwargs={"escalation_phone": escalation_phone},
        daemon=True,
    )
    t.start()
    print("Support agent polling started.")
    return t


def stop_support_agent():
    _stop_flag.set()
    print("Support agent polling stopped.")


# To run:
# agent_thread = start_support_agent(escalation_phone="+15550001234")
# # ... later ...
# stop_support_agent()

print("Production polling helpers defined.")


### 6b. Webhook-Driven Processing (Recommended for Production)

Instead of polling, configure Commune to POST a webhook when a new email arrives.
This removes the polling delay and reduces API calls.

```python
# Example Flask webhook handler
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/commune/webhook", methods=["POST"])
def handle_new_email():
    payload = request.json
    email_id = payload["email_id"]

    # Process asynchronously — don't block the webhook response
    threading.Thread(
        target=run_support_agent,
        args=(f"Process email {email_id}: read it, classify it, and reply.",),
        daemon=True,
    ).start()

    return jsonify({"status": "accepted"}), 202
```

Set your webhook URL in the [Commune dashboard](https://commune.email/dashboard).


## 7. Conclusion

You now have a complete customer support email agent that:

- Uses **GPT-4o function calling** to read, classify, and respond to support emails
- Integrates with **Commune** for real inbox access and email/SMS delivery
- Handles the full **tool_calls loop** correctly, including multi-step reasoning
- Includes **production patterns** for polling, error handling, and webhook integration

### Extensions

- Add a `create_ticket` tool to sync resolved issues to Jira or Linear
- Use [Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs)
  to get machine-readable triage metadata alongside the reply
- Store processed email IDs in Redis to avoid re-processing
- Implement a human-in-the-loop checkpoint for P0 replies before they send

### Resources

- [OpenAI Function Calling guide](https://platform.openai.com/docs/guides/function-calling)
- [Commune API reference](https://commune.email/docs)
- [commune-mail on PyPI](https://pypi.org/project/commune-mail/)
- [OpenAI Cookbook — Orchestrating agents](https://cookbook.openai.com/examples/orchestrating_agents)
